# Cognivision Voice Intelligence
## Notebook 03: Acoustic Feature Engineering & Dimensionality Reduction
This notebook analyzes extracted acoustic features (RMS energy, Zero-Crossing Rate, Spectral Centroid, Bandwidth, Contrast, and MFCCs 1–13) and performs Principal Component Analysis (PCA) to inspect 2D and 3D feature spaces.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

root_dir = Path("..").resolve()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from backend.app.services.feature_service import feature_service

### 1. Load Extracted Features

In [ ]:
feat_df = feature_service.get_features_dataframe()
if feat_df is None or len(feat_df) == 0:
    print("Extracting features...")
    feature_service.extract_dataset_features(max_samples=1200)
    feat_df = feature_service.get_features_dataframe()

print(f"Features DataFrame shape: {feat_df.shape}")
feat_df.head()

### 2. Acoustic Feature Distributions by Emotion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.boxplot(data=feat_df, x='emotion', y='rms_mean', ax=axes[0], palette='Set2')
axes[0].set_title("RMS Energy by Emotion", fontweight='bold')

sns.boxplot(data=feat_df, x='emotion', y='spectral_centroid_mean', ax=axes[1], palette='Set2')
axes[1].set_title("Spectral Centroid by Emotion", fontweight='bold')

sns.boxplot(data=feat_df, x='emotion', y='zcr_mean', ax=axes[2], palette='Set2')
axes[2].set_title("Zero Crossing Rate by Emotion", fontweight='bold')

plt.tight_layout()
plt.show()

### 3. Principal Component Analysis (PCA) 2D Projection

In [ ]:
meta_cols = ["file_path", "file_name", "emotion", "actor_id", "split"]
feat_cols = [c for c in feat_df.columns if c not in meta_cols and feat_df[c].dtype != object]

X = feat_df[feat_cols].fillna(0).values
y = feat_df['emotion'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(9, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, palette='bright', alpha=0.7)
plt.title(f"2D Acoustic Feature PCA (Explained Variance: {pca.explained_variance_ratio_.sum()*100:.1f}%)", fontweight='bold')
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.show()